In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

In [2]:
from Utils.Accuracy_measures import topk_accuracy
from Utils.TinyImageNet_loader import get_tinyimagenet_dataloaders
from Utils.Num_parameter import count_parameters
from Models.Resnet50 import Resnet50

import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [3]:
device = 'cuda'

In [4]:
model = Resnet50(pretrained=True,
                          weights_path='../weights/resnet50_weights.pth',
                          tensorized=False,
                          input_shape=(192,192),
                          num_classes=200,
                          avg_pool=False,
                          new_classifier=None).to(device)

In [5]:
model.load_state_dict(torch.load('../weights/temp.pth'))

<All keys matched successfully>

In [6]:
image_size = 192

tiny_transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(64, padding=4),
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_val = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_test = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])


tiny_train_loader, tiny_val_loader, tiny_test_loader = get_tinyimagenet_dataloaders(data_dir = '../datasets',
                                                                                    transform_train=tiny_transform_train,
                                                                                    transform_val=tiny_transform_val,
                                                                                    transform_test=tiny_transform_test,
                                                                                    batch_size=64,
                                                                                    image_size=192)

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [8]:
def test(loader, epoch):
    model.eval()
    
    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
            
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)
    
    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [9]:
def train(loader, epoch):
    model.train()
    
    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)
    
    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

In [10]:
# t1 = train(tiny_train_loader, 1)
# t2 = train(tiny_train_loader, 2)
# t3 = train(tiny_train_loader, 3)
# t4 = train(tiny_train_loader, 4)
# t5 = train(tiny_train_loader, 5)
# t6 = train(tiny_train_loader, 6)
# torch.save(model.state_dict(), '../weights/temp.pth')

In [11]:
r1 = test(tiny_train_loader, 1)

Test epoch 1: top1=0.6483599543571472%, top2=0.7687000036239624%, top3=0.8235399723052979%, top4=0.856469988822937%, top5=0.8785099983215332%, loss=0.02070767374277115, time=82.5421142578125s


In [12]:
r = test(tiny_val_loader, 1)

Test epoch 1: top1=0.002099999925121665%, top2=0.005799999926239252%, top3=0.008899999782443047%, top4=0.012399999424815178%, top5=0.01589999906718731%, loss=0.2162377861022949, time=9.517295598983765s
